In [ ]:
import requests
import os
from tqdm import tqdm  # For progress bar
import time
import pyautogui  # For preventing laptop from sleeping

def download_file(url, local_filename):
    """Helper function to download a file from given URL."""
    with requests.get(url, stream=True) as r:
        try:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            block_size = 1024  # 1 Kibibyte
            progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True)
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(block_size):
                    progress_bar.update(len(chunk))
                    f.write(chunk)
            progress_bar.close()
            return local_filename
        except requests.exceptions.HTTPError as err:
            print(f"HTTP error occurred: {err}")
            return None

# Function to prevent laptop from sleeping
def prevent_sleep():
    pyautogui.press('shift')  # Pressing Shift key to prevent sleep

def download_data(base_url, model, experiment, variant_label, variable, years_range, _update_version, grid_label, download_dir):
    """Download data files for specified parameters, years range, _update version, and grid label."""
    # Create download directory if it doesn't exist
    os.makedirs(download_dir, exist_ok=True)
    
    # Create subfolder structure based on parameters
    subfolder = os.path.join(download_dir, model, experiment, variant_label, variable)
    os.makedirs(subfolder, exist_ok=True)
    
    # Loop through each year in the range
    for year in range(years_range[0], years_range[1] + 1):
        # Construct filename and URL for each year
        filename = f"{variable}_day_{model}_{experiment}_{variant_label}_{grid_label}_{year}{_update_version}.nc"
        url = f"{base_url}/{model}/{experiment}/{variant_label}/{variable}/{filename}"
        
        # Download data file
        print(f"Downloading {filename}...")
        local_filename = os.path.join(subfolder, filename)
        downloaded_file = download_file(url, local_filename)
        
        if downloaded_file:
            print(f"Downloaded {filename}")
        else:
            print(f"Failed to download {filename}")
        
        # Prevent laptop from sleeping after each download
        prevent_sleep()
    
    print("Download complete!")

if __name__ == "__main__":
    # Example usage:
    
    # Base URL for NEX-GDDP CMIP6 dataset
    base_url = "https://nex-gddp-cmip6.s3-us-west-2.amazonaws.com/NEX-GDDP-CMIP6"
    
    # Parameters to specify (customize these)
    model = input("Enter model name (e.g., CanESM5, INM-CM5-0): ")         # Model name
    experiment = input("Enter experiment ID (e.g., historical, SSP126, ssp245, ssp370, ssp585): ") # Experiment ID
    variant_label = input("Enter variant label (e.g., r1i1p1f1, r4i1p1f1, r3i1p1f1, r1i1p1f2, r1i1p1f3): ")         # Variant label
    variable = input("Enter variable name (e.g., pr, tas, tasmax, tasmin): ")                    # Variable name
    grid_label = input("Enter grid label (e.g., gn, gr, gr1): ")            # Grid label
    start_year = int(input("Enter start year: 1950, 2015"))                           # Start year
    end_year = int(input("Enter end year: 2014, 2100"))                               # End year
    update_version = input("Enter update version (e.g., , _v1.1, _v1.2): ")           # Update version
    
    years_range = (start_year, end_year)  # Range of years to download
    
    # Directory to save downloads
    download_dir = "./nex_gddp_cmip6_data"
    
    # Download data
    download_data(base_url, model, experiment, variant_label, variable, years_range, update_version, grid_label, download_dir)
